In [9]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent))

from config import DOSSIER_CP
from src.extraction_cp.extraction import analyser_cp
from src.retrieval.retrieval import ouvrir_store, rechercher

store = ouvrir_store()
texte_cp, fiche, contrat_id = analyser_cp(DOSSIER_CP / "cp_01_luko_etendue.pdf")
question = "Suis-je couvert si on me vole mon vélo dans ma cave ?"
passages = rechercher(store, question, contrat_id)
print(contrat_id, "|", len(passages), "passages")

luko_hce_2504 | 5 passages


In [10]:
def formater_passages(passages):
    """Numérote chaque passage et indique sa source, pour que le LLM puisse la citer."""
    blocs = []
    for i, p in enumerate(passages, start=1):
        m = p.metadata
        blocs.append(f"[{i}] (CG {m['assureur']}, {m['fichier']}, page {m['page']})\n{p.page_content}")
    return "\n\n".join(blocs)


PROMPT = """Tu es un assistant qui aide un assuré à comprendre SON contrat d'assurance habitation.

Règles :
1. Réponds UNIQUEMENT à partir des documents ci-dessous. N'utilise pas tes connaissances générales.
2. Les Conditions Particulières (CP) sont propres au client et PRÉVALENT sur les Conditions Générales (CG) en cas de contradiction.
3. Cite tes sources après chaque information : [CP] pour les conditions particulières, [1], [2]... pour les extraits des CG.
4. Si les documents ne permettent pas de répondre, dis-le clairement : « Je ne trouve pas cette information dans vos documents. »
5. Réponds en français, en 3 à 6 phrases, simplement.

=== CONDITIONS PARTICULIÈRES DU CLIENT ===
{cp}

=== EXTRAITS DES CONDITIONS GÉNÉRALES ===
{passages}

=== QUESTION ===
{question}

Réponse :"""

prompt = PROMPT.format(cp=texte_cp, passages=formater_passages(passages), question=question)
print(prompt)
print(f"\n{len(prompt)} caractères, environ {len(prompt) / 3.5:.0f} tokens")

Tu es un assistant qui aide un assuré à comprendre SON contrat d'assurance habitation.

Règles :
1. Réponds UNIQUEMENT à partir des documents ci-dessous. N'utilise pas tes connaissances générales.
2. Les Conditions Particulières (CP) sont propres au client et PRÉVALENT sur les Conditions Générales (CG) en cas de contradiction.
3. Cite tes sources après chaque information : [CP] pour les conditions particulières, [1], [2]... pour les extraits des CG.
4. Si les documents ne permettent pas de répondre, dis-le clairement : « Je ne trouve pas cette information dans vos documents. »
5. Réponds en français, en 3 à 6 phrases, simplement.

=== CONDITIONS PARTICULIÈRES DU CLIENT ===
DOCUMENT FICTIF - créé pour un projet pédagogique de RAG. Aucune valeur contractuelle.
CONDITIONS PARTICULIÈRES - Assurance Habitation
Assureur :
Luko by Allianz Direct
N° de contrat :
LK-HAB-2025-004817
Souscripteur :
Mme Camille DURAND (personne fictive)
Date d'effet :
01/06/2025
Le présent contrat est régi par le 

In [12]:
import time
from langchain_ollama import ChatOllama

llm = ChatOllama(model=LLM_MODEL, temperature=0, num_ctx=8192, num_predict=600)

debut = time.time()
reponse = llm.invoke(prompt)
print(reponse.content)
print(f"\nTemps : {time.time() - debut:.1f} s")
print("Tokens :", reponse.usage_metadata)

Oui, vous êtes couvert pour le vol de votre vélo situé dans votre cave, car les Conditions Générales mentionnent que la garantie couvre le vol de vos biens commis à l'intérieur du logement assuré, y compris par effraction des locaux [3].

Temps : 11.3 s
Tokens : {'input_tokens': 1882, 'output_tokens': 67, 'total_tokens': 1949}


In [5]:
import re

def ajouter_sources(texte_reponse, passages):
    """Remplace les numéros cités par le LLM par les vraies sources (fichier + page)."""
    numeros = sorted({int(n) for n in re.findall(r"\[(\d+)\]", texte_reponse)})
    lignes = []
    if "[CP]" in texte_reponse:
        lignes.append("[CP] Vos Conditions Particulières")
    for n in numeros:
        if 1 <= n <= len(passages):
            m = passages[n - 1].metadata
            lignes.append(f"[{n}] {m['fichier']}, page {m['page']}")
        else:
            lignes.append(f"[{n}] ⚠️ citation inexistante")
    return texte_reponse + ("\n\nSources :\n" + "\n".join(lignes) if lignes else "")

In [6]:
PROMPT = PROMPT.replace(
    "5. Réponds en français, en 3 à 6 phrases, simplement.",
    """5. Réponds en français, avec exactement cette structure :
Réponse : oui / non / oui sous conditions / je ne sais pas, en une phrase.
Votre contrat : ce que prévoient les CP (garantie souscrite ou non, franchise, plafonds) [CP].
Conditions et exclusions : ce que disent les CG, avec leurs numéros [1], [2]...
6. Ne conclus « oui » que si un extrait le dit clairement. Une cave, un garage ou un grenier sont des dépendances, pas l'intérieur du logement.""")

prompt = PROMPT.format(cp=texte_cp, passages=formater_passages(passages), question=question)
reponse = llm.invoke(prompt)
print(ajouter_sources(reponse.content, passages))

Non, je ne trouve pas cette information dans vos documents.


In [7]:
PROMPT = PROMPT.replace(
    "4. Si les documents ne permettent pas de répondre, dis-le clairement : « Je ne trouve pas cette information dans vos documents. »",
    """4. Si les documents ne parlent PAS DU TOUT du sujet, réponds : « Je ne trouve pas cette information dans vos documents. »
   S'ils y répondent EN PARTIE, donne ce qu'ils disent avec les sources, puis précise ce qui n'y figure pas.""")

PROMPT = PROMPT.replace(
    "6. Ne conclus « oui » que si un extrait le dit clairement. Une cave, un garage ou un grenier sont des dépendances, pas l'intérieur du logement.",
    "6. Ne conclus « oui » que si un extrait le dit clairement. Sinon, réponds « oui sous conditions » ou « à vérifier ».")

prompt = PROMPT.format(cp=texte_cp, passages=formater_passages(passages), question=question)
print(prompt[:1500])          # vérifie que les règles ont bien été remplacées
reponse = llm.invoke(prompt)
print(ajouter_sources(reponse.content, passages))

Tu es un assistant qui aide un assuré à comprendre SON contrat d'assurance habitation.

Règles :
1. Réponds UNIQUEMENT à partir des documents ci-dessous. N'utilise pas tes connaissances générales.
2. Les Conditions Particulières (CP) sont propres au client et PRÉVALENT sur les Conditions Générales (CG) en cas de contradiction.
3. Cite tes sources après chaque information : [CP] pour les conditions particulières, [1], [2]... pour les extraits des CG.
4. Si les documents ne parlent PAS DU TOUT du sujet, réponds : « Je ne trouve pas cette information dans vos documents. »
   S'ils y répondent EN PARTIE, donne ce qu'ils disent avec les sources, puis précise ce qui n'y figure pas.
5. Réponds en français, avec exactement cette structure :
Réponse : oui / non / oui sous conditions / je ne sais pas, en une phrase.
Votre contrat : ce que prévoient les CP (garantie souscrite ou non, franchise, plafonds) [CP].
Conditions et exclusions : ce que disent les CG, avec leurs numéros [1], [2]...
6. Ne c

In [8]:
PROMPT = """Tu es un assistant qui aide un assuré à comprendre SON contrat d'assurance habitation.

Règles :
1. Réponds UNIQUEMENT à partir des documents ci-dessous. N'utilise pas tes connaissances générales.
2. Les Conditions Particulières (CP) sont propres au client et PRÉVALENT sur les Conditions Générales (CG).
3. Cite tes sources après chaque information : [CP] pour les conditions particulières, [1], [2]... pour les extraits des CG.
4. Si les documents ne parlent PAS DU TOUT du sujet, réponds : « Je ne trouve pas cette information dans vos documents. »
   S'ils y répondent EN PARTIE, donne ce qu'ils disent avec les sources, puis précise ce qui n'y figure pas.
5. Ne conclus « oui » que si un extrait le dit clairement. Sinon, réponds « oui sous conditions » ou « à vérifier ».
6. Réponds en français, avec exactement cette structure :
Réponse : oui / non / oui sous conditions / à vérifier / je ne sais pas, en une phrase.
Votre contrat : ce que prévoient les CP (garantie souscrite ou non, franchise, plafonds) [CP].
Conditions et exclusions : ce que disent les CG, avec leurs numéros [1], [2]...

=== CONDITIONS PARTICULIÈRES DU CLIENT ===
{cp}

=== EXTRAITS DES CONDITIONS GÉNÉRALES ===
{passages}

=== QUESTION ===
{question}
"""

prompt = PROMPT.format(cp=texte_cp, passages=formater_passages(passages), question=question)
assert "EN PARTIE" in prompt          # on vérifie que c'est bien la nouvelle version

for modele, options in [("qwen2.5:7b", {}), ("qwen3:8b", {"reasoning": False})]:
    llm_test = ChatOllama(model=modele, temperature=0, num_ctx=8192, num_predict=600, **options)
    debut = time.time()
    rep = llm_test.invoke(prompt)
    print(f"===== {modele} ({time.time() - debut:.1f} s) =====")
    print(ajouter_sources(rep.content, passages), "\n")

===== qwen2.5:7b (6.7 s) =====
Réponse : oui / non / oui sous conditions / à vérifier / je ne sais pas, en une phrase.

Souscrite la garantie responsabilité civile vie privée, mais les CG ne mentionnent pas spécifiquement la couverture pour le vol de vélos dans la cave, donc à vérifier. [1]

Sources :
[1] luko_habitation-etendue_cg_2504.pdf, page 47 

===== qwen3:8b (25.6 s) =====
Réponse : oui sous conditions.  
Votre contrat couvre le vol de votre vélo dans la cave, à condition que le vélo soit assuré dans le cadre de la garantie Vol et Vandalisme [CP]. Les Conditions Générales indiquent que le vol des biens commis à l'intérieur du logement assuré est couvert, sous réserve d'un niveau de protection approprié [3]. Cependant, les Conditions Générales ne mentionnent pas explicitement la couverture du vélo dans la cave, mais la cave fait partie du logement assuré [1]. Il convient de vérifier si le vélo est considéré comme un bien assuré dans le cadre de la garantie Vol.

Sources :
[CP] V